Name: Nik Amir Faris Bin Nik Amidi
Student ID : SW01083709
Section : 01BT

In [1]:
pip install pandas numpy matplotlib seaborn scikit-learn nltk wordcloud

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [2]:
#STEP 1: Load Dataset
import pandas as pd

df = pd.read_csv('Reviews.csv')

print(df.head())
print(df.shape)

   Id   ProductId          UserId                      ProfileName  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW                       delmartian   
1   2  B00813GRG4  A1D87F6ZCVE5NK                           dll pa   
2   3  B000LQOCH0   ABXLMWJIXXAIN  Natalia Corres "Natalia Corres"   
3   4  B000UA0QIQ  A395BORC6FGVXV                             Karl   
4   5  B006K2ZZ7K  A1UQRSCLF8GW1T    Michael D. Bigham "M. Wassir"   

   HelpfulnessNumerator  HelpfulnessDenominator  Score        Time  \
0                     1                       1      5  1303862400   
1                     0                       0      1  1346976000   
2                     1                       1      4  1219017600   
3                     3                       3      2  1307923200   
4                     0                       0      5  1350777600   

                 Summary                                               Text  
0  Good Quality Dog Food  I have bought several of the Vitality canned d...  
1 

In [3]:
#STEP 2: Data Preprocessing

df = df[['Score', 'Text']]
def sentiment_label(score):
    if score >= 4:
        return 'positive'
    elif score <= 2:
        return 'negative'
    else:
        return 'neutral'

df['Sentiment'] = df['Score'].apply(sentiment_label)
df = df[df['Sentiment'] != 'neutral']

#Text Cleaning
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

df['clean_text'] = df['Text'].apply(clean_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/62af69a0-0330-4246-8917-
[nltk_data]     f16436100860/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
#STEP 3: Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['clean_text'])
y = df['Sentiment']

In [5]:
#STEP 4: Train-Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
#STEP 5: Model Selection (VERY IMPORTANT PART)

#5A: Machine Learning Models

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

In [7]:
#Step 5.5 Vader Prediction
from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    score = sia.polarity_scores(text)['compound']
    if score >= 0:
        return 'positive'
    else:
        return 'negative'

df['vader_pred'] = df['clean_text'].apply(vader_sentiment)

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/62af69a0-0330-4246-8917-
[nltk_data]     f16436100860/nltk_data...


In [8]:
#STEP 6: Model Evaluation

#6.1 Metrics
from sklearn.metrics import accuracy_score, classification_report

print("Logistic Regression")
print(accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

print("Naive Bayes")
print(accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

#6.2 Evaluate VADER
print("VADER Accuracy")
print(accuracy_score(df['Sentiment'], df['vader_pred']))

Logistic Regression
0.8713333333333333
              precision    recall  f1-score   support

    negative       0.95      0.25      0.39       253
    positive       0.87      1.00      0.93      1247

    accuracy                           0.87      1500
   macro avg       0.91      0.62      0.66      1500
weighted avg       0.88      0.87      0.84      1500

Naive Bayes
0.8393333333333334
              precision    recall  f1-score   support

    negative       0.93      0.05      0.10       253
    positive       0.84      1.00      0.91      1247

    accuracy                           0.84      1500
   macro avg       0.88      0.53      0.50      1500
weighted avg       0.85      0.84      0.77      1500

VADER Accuracy
0.8612778444711218


Machine learning models such as Logistic Regression and Naive Bayes performed better compared to the lexicon-based VADER approach. These models can learn patterns from data and adapt to context, making them more accurate for sentiment classification. However, they require training data and computational resources. On the other hand, VADER is fast and does not require training, but it struggles with sarcasm and complex sentence structures. Overall, machine learning approaches are more suitable for large-scale sentiment analysis tasks.



In [9]:
df.to_csv('processed_reviews.csv', index=False)